# 03 – Transformer: distilbert-base-multilingual-cased (Google Colab, GPU)

**Dieses Notebook läuft NUR auf Google Colab mit GPU** (das Modell ist ~540 MB, Training auf CPU dauert Stunden).

## So öffnest du es in Colab

1. Öffne diesen Link (Branch ggf. anpassen):
   `https://colab.research.google.com/github/jhyoun2028/BWKI_dp/blob/main/notebooks/03_transformer.ipynb`
   – oder in Colab: *Datei → Notebook öffnen → GitHub* → `jhyoun2028/BWKI_dp` → `notebooks/03_transformer.ipynb`.
2. *Laufzeit → Laufzeittyp ändern → T4 GPU* auswählen und speichern.
3. Optional, damit die Ergebnisse zurück ins Repo gepusht werden: in Colab links das
   Schlüssel-Symbol (*Secrets*) öffnen und ein Secret `GITHUB_TOKEN` (GitHub Personal Access Token
   mit `repo`-Recht) anlegen. Ohne Token werden die Ergebnisdateien nur auf Google Drive gespeichert.
4. *Laufzeit → Alle ausführen*. Beim Drive-Mount einmal bestätigen. Dauer: ca. 10–15 Minuten.

**Was passiert:** Repo klonen → dieselben Splits wie die Baseline laden → DistilBERT 3 Epochen
feintunen (bestes Checkpoint nach Val-F1) → **genau eine** Auswertung auf `test.csv` →
`results/` aktualisieren → Modell nach Google Drive `DoppelCheck/models/distilbert/` speichern →
Vergleichstabelle Baseline vs. DistilBERT → `results/` committen und pushen.

In [1]:
# --- 1. Konfiguration, Google Drive, Repo klonen, Pakete installieren -----------------
REPO = "jhyoun2028/BWKI_dp"
BRANCH = "main"            # ggf. auf den Arbeits-Branch setzen
REPO_DIR = "/content/BWKI_dp"
DRIVE_MODEL_DIR = "/content/drive/MyDrive/DoppelCheck/models/distilbert"

import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = None
clone_url = (f"https://{GITHUB_TOKEN}@github.com/{REPO}.git" if GITHUB_TOKEN
             else f"https://github.com/{REPO}.git")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, clone_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Repo:", REPO_DIR, "| Branch:", BRANCH, "| Token vorhanden:", bool(GITHUB_TOKEN))

Mounted at /content/drive
Repo: /content/BWKI_dp | Branch: main | Token vorhanden: False


In [2]:
%pip install -q transformers datasets accelerate scikit-learn tldextract
# torch ist auf Colab vorinstalliert; nur nachinstallieren, wenn es fehlt.
import importlib.util
if importlib.util.find_spec("torch") is None:
    %pip install -q torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 2.1 MB/s eta 0:00:00


In [3]:
# --- 2. GPU prüfen, Seeds setzen ------------------------------------------------------
import torch, transformers, numpy as np, random
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "Keine GPU! Laufzeit → Laufzeittyp ändern → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
transformers.set_seed(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

torch 2.11.0+cu128 | transformers 5.16.1
GPU: Tesla T4


In [4]:
# --- 3. Dieselben Splits wie die Baseline laden (NIE neu splitten) ---------------------
import pandas as pd
from datasets import Dataset

train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")
test_df = pd.read_csv("data/processed/test.csv")
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: {len(d)} Zeilen, spam-Anteil {d.label.mean():.3f}, de-Zeilen {(d.lang == 'de').sum()}")

MODEL_NAME = "distilbert-base-multilingual-cased"
MAX_LEN = 128
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(df):
    ds = Dataset.from_pandas(df[["text", "label"]].astype({"text": str, "label": int}), preserve_index=False)
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)

train_ds, val_ds, test_ds = to_dataset(train_df), to_dataset(val_df), to_dataset(test_df)

train: 4408 Zeilen, spam-Anteil 0.150, de-Zeilen 304
val: 513 Zeilen, spam-Anteil 0.123, de-Zeilen 0
test: 617 Zeilen, spam-Anteil 0.241, de-Zeilen 104


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/4408 [00:00<?, ? examples/s]

Map:   0%|          | 0/513 [00:00<?, ? examples/s]

Map:   0%|          | 0/617 [00:00<?, ? examples/s]

In [ ]:
# --- 4. Modell + Trainer (3 Epochen, lr 2e-5, batch 16, bestes Checkpoint nach Val-F1) --
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds),
            "precision": precision_score(labels, preds, zero_division=0),
            "recall": recall_score(labels, preds, zero_division=0),
            "f1": f1_score(labels, preds, zero_division=0)}

args = TrainingArguments(
    output_dir="/content/ckpt_distilbert",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    seed=SEED,
    fp16=True,
    logging_steps=50,
    report_to="none",
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
# --- 5. Training ----------------------------------------------------------------------
train_result = trainer.train()
print(train_result.metrics)
val_history = [h for h in trainer.state.log_history if "eval_f1" in h]
for h in val_history:
    print(f"Epoche {h['epoch']:.0f}: val F1 = {h['eval_f1']:.4f}, val acc = {h['eval_accuracy']:.4f}")
best_val_f1 = max(h["eval_f1"] for h in val_history)
print("Bestes Val-F1:", round(best_val_f1, 4), "(dieses Checkpoint ist geladen)")

In [ ]:
# --- 6. Auswertung: train und GENAU EINMAL test --------------------------------------
from sklearn.metrics import confusion_matrix

def metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = (int(v) for v in cm.ravel())
    return {"n": int(len(y_true)),
            "accuracy": round(accuracy_score(y_true, y_pred), 4),
            "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
            "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
            "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
            "fpr": round(fp / (fp + tn), 4) if (fp + tn) else None,
            "tn": tn, "fp": fp, "fn": fn, "tp": tp}

def predict(ds):
    out = trainer.predict(ds)
    probs = torch.softmax(torch.tensor(out.predictions), dim=-1)[:, 1].numpy()
    return out.predictions.argmax(-1), probs

train_pred, _ = predict(train_ds)
test_pred, test_prob = predict(test_ds)          # einzige Test-Auswertung
m_train = metrics(train_df["label"], train_pred)
m_test = metrics(test_df["label"], test_pred)
de = (test_df["lang"] == "de").to_numpy()
m_test_de = metrics(test_df.loc[de, "label"], test_pred[de]) if de.any() else None

print("TRAIN:", m_train)
print("TEST :", m_test)
print("TEST (nur Deutsch):", m_test_de if m_test_de else "keine deutschen Zeilen im Testset")

In [ ]:
# --- 7. results/ aktualisieren: metrics.json, experiments.csv, Konfusionsmatrix --------
import json, csv
from datetime import date
from pathlib import Path
import matplotlib.pyplot as plt

RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
KEY = "distilbert_multilingual"
params = {"model": MODEL_NAME, "epochs": 3, "lr": 2e-5, "batch": 16, "max_len": MAX_LEN,
          "best_checkpoint_by": "val_f1", "fp16": True}
entry = {"date": date.today().isoformat(), "seed": SEED, "params": params,
         "val_f1_best": round(float(best_val_f1), 4), "train": m_train, "test": m_test,
         "test_de": m_test_de,
         "misclassified_test": [
             {"text": t, "label": int(l), "pred": int(p), "p_phishing": round(float(pr), 4), "lang": lg}
             for t, l, p, pr, lg in zip(test_df.text, test_df.label, test_pred, test_prob, test_df.lang)
             if int(l) != int(p)]}

mp = RESULTS / "metrics.json"
all_metrics = json.loads(mp.read_text(encoding="utf-8")) if mp.exists() else {}
all_metrics[KEY] = entry                       # Baseline-Schlüssel bleibt erhalten
mp.write_text(json.dumps(all_metrics, indent=2, ensure_ascii=False), encoding="utf-8")

ep = RESULTS / "experiments.csv"
new_file = not ep.exists()
with ep.open("a", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    if new_file:
        w.writerow(["date", "model", "params", "seed", "train_acc", "test_acc", "f1", "fpr"])
    w.writerow([entry["date"], KEY, json.dumps(params), SEED,
                m_train["accuracy"], m_test["accuracy"], m_test["f1"], m_test["fpr"]])

cm = confusion_matrix(test_df["label"], test_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(4.2, 3.8))
im = ax.imshow(cm, cmap="Blues", vmin=0)
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, f"{v:,}", ha="center", va="center",
            color="white" if v > cm.max() * 0.6 else "#1f2933", fontsize=12)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["ham (0)", "phishing (1)"]); ax.set_yticklabels(["ham (0)", "phishing (1)"])
ax.set_xlabel("vorhergesagt"); ax.set_ylabel("tatsächlich")
ax.set_title("DistilBERT multilingual – Testdaten", fontsize=11)
for s in ax.spines.values():
    s.set_visible(False)
ax.tick_params(length=0)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout(); fig.savefig(RESULTS / "confusion_matrix_distilbert.png", dpi=150); plt.show()
print("gespeichert:", mp, ep, RESULTS / "confusion_matrix_distilbert.png")

In [ ]:
# --- 8. Modell nach Google Drive speichern (zu groß für das Code-ZIP) ------------------
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
trainer.save_model(DRIVE_MODEL_DIR)
tokenizer.save_pretrained(DRIVE_MODEL_DIR)
size_mb = sum(p.stat().st_size for p in Path(DRIVE_MODEL_DIR).rglob("*") if p.is_file()) / 1e6
print(f"Modell gespeichert unter {DRIVE_MODEL_DIR}: {size_mb:.0f} MB")
print("→ Ordner in Drive freigeben (Link) und im README verlinken; lokal nach models/distilbert/ kopieren.")

In [ ]:
# --- 9. Pro-Zeile-Wahrscheinlichkeiten + Schwellenwert-Tuning (wie bei der Baseline) -----
# Nutzt das noch im Speicher liegende feingetunte Modell. Das Modell selbst (~540 MB) kommt
# NICHT ins Repo; hier wird alles gespeichert, womit man ohne Modell weiterrechnen kann.
import sys
sys.path.insert(0, "src")
from tune_threshold import MAX_FPR, THRESHOLDS, append_entry, tune   # exakt dieselbe Logik wie für die Baseline

# (1) eine Zeile je Test-Beispiel -> results/distilbert_test_probs.csv
probs_df = pd.DataFrame({
    "index": test_df.index,
    "text": test_df["text"],
    "lang": test_df["lang"],
    "label": test_df["label"].astype(int),
    "p_phishing": np.round(test_prob, 6),
})
probs_path = RESULTS / "distilbert_test_probs.csv"
probs_df.to_csv(probs_path, index=False)
print(f"{probs_path}: {len(probs_df)} Zeilen (davon de: {int((probs_df.lang == 'de').sum())})")
print("→ damit lässt sich die Ampel-Tabelle ohne das Modell neu rechnen:")
print("   python src/eval_pipeline.py --probs results/distilbert_test_probs.csv")

# (2) Schwellenwert NUR auf der Validierung suchen; die Testdaten bleiben unangetastet
_, val_prob = predict(val_ds)
_, train_prob = predict(train_ds)
thr, m_val = tune(val_prob, val_df["label"].to_numpy())
print(f"\nSweep {THRESHOLDS[0]}–{THRESHOLDS[-1]} auf val, Kriterium: max recall bei FPR <= {MAX_FPR}")
print(f"Gewählter Schwellenwert: {thr:.2f}  (val: recall {m_val['recall']:.4f}, FPR {m_val['fpr']:.4f})")

pred_thr = (test_prob >= thr).astype(int)
m_train_thr = metrics(train_df["label"], (train_prob >= thr).astype(int))
m_test_thr = metrics(test_df["label"], pred_thr)
m_de_thr = metrics(test_df.loc[de, "label"], pred_thr[de]) if de.any() else None
print("TEST      @thr:", m_test_thr)
print("TEST (de) @thr:", m_de_thr if m_de_thr else "keine deutschen Zeilen")
print("TEST (de) @0.50:", m_test_de if m_test_de else "keine deutschen Zeilen")

append_entry(f"{KEY}_thr{thr:.2f}", {
    "date": date.today().isoformat(), "seed": SEED,
    "params": {"base_model": KEY, "threshold": round(float(thr), 2), "tuned_on": "val (English only)",
               "criterion": f"max recall s.t. FPR <= {MAX_FPR}",
               "val_at_threshold": {k: m_val[k] for k in ("recall", "fpr", "f1")}},
    "train": m_train_thr, "test": m_test_thr, "test_de": m_de_thr})
print(f"\n'{KEY}_thr{thr:.2f}' an results/metrics.json und results/experiments.csv angehängt")

In [ ]:
# --- 10. Vergleich Baseline vs. DistilBERT + results/ zurück ins Repo ------------------
m = json.loads(mp.read_text(encoding="utf-8"))
rows = []
for key in ["baseline_tfidf_logreg", KEY]:
    if key in m:
        e = m[key]
        rows.append({"model": key,
                     "train_acc": e["train"]["accuracy"], "test_acc": e["test"]["accuracy"],
                     "test_f1": e["test"]["f1"], "test_fpr": e["test"]["fpr"],
                     "test_de_f1": e["test_de"]["f1"] if e.get("test_de") else None})
display(pd.DataFrame(rows).set_index("model"))

if GITHUB_TOKEN:
    subprocess.run(["git", "config", "user.email", "colab@doppelcheck.local"], check=True)
    subprocess.run(["git", "config", "user.name", "DoppelCheck Colab"], check=True)
    subprocess.run(["git", "add", "results/"], check=True)
    r = subprocess.run(["git", "commit", "-m", "step 4: distilbert results from Colab"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    r = subprocess.run(["git", "push", "origin", f"HEAD:{BRANCH}"], capture_output=True, text=True)
    print("push:", "ok" if r.returncode == 0 else r.stderr)
else:
    backup = "/content/drive/MyDrive/DoppelCheck/results"
    os.makedirs(backup, exist_ok=True)
    subprocess.run(["cp", "-r", "results/.", backup], check=True)
    print("Kein GITHUB_TOKEN-Secret: results/ nach", backup, "kopiert – bitte manuell ins Repo übernehmen.")